# 🍔 EDA Assignment — Food Delivery Dataset Analysis
### Module 02 | neural-nomad-ai-lab

**Scenario:** You are a data analyst at a food delivery startup. The operations team has handed you a raw dataset of 350 orders. It is messy — inconsistent types, dirty values, missing data, and encoding issues. Your job is to clean it, explore it, and surface actionable insights.

**Dataset:** 350 orders across Indian cities with columns for customer info, order details, payment, delivery time, and ratings.

**What this covers:**
- Data inspection & quality audit
- Data cleaning (dirty types, inconsistent categories, negative values)
- Missing value treatment
- Univariate & bivariate analysis
- Outlier detection using IQR
- Feature engineering
- Business insights

In [1]:
import pandas as pd
import numpy as np

base = "https://raw.githubusercontent.com/neuralnomad26/neural-nomad-ai-lab/main/modules/02-eda/datasets/"
df = pd.read_csv(base + "food_delivery_dataset.csv")

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

Shape: (350, 16)
Columns: ['order_id', 'customer_name', 'customer_age', 'customer_city', 'order_date', 'delivery_timestamp', 'restaurant_name', 'food_item', 'quantity', 'price', 'discount', 'payment_method', 'delivery_rating', 'special_instructions', 'order_status', 'delivery_time_minutes']


## Section 1 — Data Inspection & Quality Audit
Before touching anything, understand exactly what you're working with.

In [2]:
# Task 1.1 — full structure overview
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 16 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               350 non-null    int64  
 1   customer_name          350 non-null    object 
 2   customer_age           246 non-null    object 
 3   customer_city          318 non-null    object 
 4   order_date             350 non-null    object 
 5   delivery_timestamp     177 non-null    object 
 6   restaurant_name        350 non-null    object 
 7   food_item              350 non-null    object 
 8   quantity               302 non-null    object 
 9   price                  170 non-null    float64
 10  discount               247 non-null    object 
 11  payment_method         350 non-null    object 
 12  delivery_rating        296 non-null    float64
 13  special_instructions   274 non-null    object 
 14  order_status           350 non-null    object 
 15  delive

In [3]:
# Task 1.2 — missing value audit
missing = pd.DataFrame({
    'count': df.isnull().sum(),
    'pct':   (df.isnull().sum() / len(df) * 100).round(1)
}).sort_values('pct', ascending=False)

print("Missing value audit:")
print(missing[missing['count'] > 0])

Missing value audit:
                         count   pct
price                      180  51.4
delivery_timestamp         173  49.4
delivery_time_minutes      141  40.3
customer_age               104  29.7
discount                   103  29.4
special_instructions        76  21.7
delivery_rating             54  15.4
quantity                    48  13.7
customer_city               32   9.1


In [4]:
# Task 1.3 — spot dirty values before cleaning
print("=== Dirty values spotted ===")
print("\ncustomer_age (top values):")
print(df['customer_age'].value_counts().head(3))

print("\nquantity (top values):")
print(df['quantity'].value_counts().head(4))

print("\ndiscount (top values):")
print(df['discount'].value_counts().head(4))

print("\norder_status:")
print(df['order_status'].value_counts())

print("\npayment_method:")
print(df['payment_method'].value_counts())

print("\ncustomer_city:")
print(df['customer_city'].value_counts())

print(f"\nNegative delivery times: {(df['delivery_time_minutes'] < 0).sum()} rows (value: -5.0)")
print(f"Rating > 5 (invalid): {(df['delivery_rating'] > 5).sum()} rows (value: 6.0)")

=== Dirty values spotted ===

customer_age (top values):
twenty    93
45.0      67
29         7
Name: count, dtype: int64

quantity (top values):
2        130
1         60
3         59
three     53
Name: count, dtype: int64

discount (top values):
10    70
15    63
5     59
5%    55
Name: count, dtype: int64

order_status:
Cancelled    80
Returned     69
completed    69
cancelled    66
Completed    66
Name: count, dtype: int64

payment_method:
Card      72
wallet    72
Cash      60
UPI       56
upi       46
CASH      44
Name: count, dtype: int64

customer_city:
Pune         48
Mumbai       47
Nagpur       46
Hyderabad    45
mumbai       35
Bangalore    33
DELHI        33
delhi        31
Name: count, dtype: int64

Negative delivery times: 98 rows (value: -5.0)
Rating > 5 (invalid): 47 rows (value: 6.0)


## Section 2 — Data Cleaning
Fix every data quality issue identified above. Each fix is intentional and documented.

In [5]:
# Task 2.1 — fix customer_age (word numbers → integers)
df_clean = df.copy()

word_to_num = {'twenty': 20, 'thirty': 30, 'forty': 40, 'fifty': 50, 'sixty': 60}
df_clean['customer_age'] = df_clean['customer_age'].replace(word_to_num)
df_clean['customer_age'] = pd.to_numeric(df_clean['customer_age'], errors='coerce')

print("customer_age — after fix:")
print(df_clean['customer_age'].describe().round(2))

customer_age — after fix:
count    246.00
mean      33.10
std       13.18
min       16.00
max       60.00
Name: customer_age, dtype: float64


In [6]:
# Task 2.2 — fix quantity and discount
df_clean['quantity'] = df_clean['quantity'].replace({'one': 1, 'two': 2, 'three': 3})
df_clean['quantity'] = pd.to_numeric(df_clean['quantity'], errors='coerce')

# Remove % sign from discount and convert
df_clean['discount'] = df_clean['discount'].astype(str).str.replace('%', '', regex=False)
df_clean['discount'] = pd.to_numeric(df_clean['discount'], errors='coerce')

print("quantity — after fix:", sorted(df_clean['quantity'].dropna().unique().tolist()))
print("discount — after fix:", sorted(df_clean['discount'].dropna().unique().tolist()))

quantity — after fix: [1.0, 2.0, 3.0]
discount — after fix: [5.0, 10.0, 15.0]


In [7]:
# Task 2.3 — standardise categorical columns (case inconsistency)
df_clean['order_status']   = df_clean['order_status'].str.strip().str.capitalize()
df_clean['payment_method'] = df_clean['payment_method'].str.strip().str.capitalize()
df_clean['customer_city']  = df_clean['customer_city'].str.strip().str.capitalize()

print("order_status — after standardising:")
print(df_clean['order_status'].value_counts())

print("\npayment_method — after standardising:")
print(df_clean['payment_method'].value_counts())

print("\ncustomer_city — after standardising:")
print(df_clean['customer_city'].value_counts())

order_status — after standardising:
Cancelled    146
Completed    135
Returned      69
Name: count, dtype: int64

payment_method — after standardising:
Card      72
Wallet    72
Cash     104
UPI      102
Name: count, dtype: int64

customer_city — after standardising:
Mumbai       82
Pune         48
Nagpur       46
Hyderabad    45
Bangalore    33
Delhi        64
Name: count, dtype: int64


In [8]:
# Task 2.4 — fix invalid numeric values
neg_count = (df_clean['delivery_time_minutes'] < 0).sum()
df_clean['delivery_time_minutes'] = df_clean['delivery_time_minutes'].apply(
    lambda x: np.nan if pd.notna(x) and x < 0 else x
)

invalid_rating = (df_clean['delivery_rating'] > 5).sum()
df_clean['delivery_rating'] = df_clean['delivery_rating'].apply(
    lambda x: np.nan if pd.notna(x) and x > 5 else x
)

print(f"Negative delivery times replaced with NaN: {neg_count} rows fixed")
print(f"Invalid ratings (>5) replaced with NaN: {invalid_rating} rows fixed")
print(f"\ndelivery_time_minutes range: {df_clean['delivery_time_minutes'].min()} — {df_clean['delivery_time_minutes'].max()} minutes")
print(f"delivery_rating range: {df_clean['delivery_rating'].min()} — {df_clean['delivery_rating'].max()}")

Negative delivery times replaced with NaN: 98 rows fixed
Invalid ratings (>5) replaced with NaN: 47 rows fixed

delivery_time_minutes range: 10.0 — 89.0 minutes
delivery_rating range: 0.0 — 5.0


In [9]:
# Task 2.5 — confirm clean data
cols = ['customer_age', 'quantity', 'price', 'discount', 'delivery_time_minutes', 'delivery_rating']
print("Clean dataset summary:")
print(df_clean[cols].describe().round(2).to_string())

Clean dataset summary:
       customer_age  quantity   price  discount  delivery_time_minutes  delivery_rating
count        246.00    302.00  170.00    192.00                 111.00           249.00
mean          33.10      2.17  250.73     10.10                  51.53             2.57
std           13.18      0.74   93.48      3.99                  22.10             1.68
min           16.00      1.00  105.95      5.00                  10.00             0.00
25%           20.00      2.00  200.00      5.00                  36.50             1.00
50%           29.00      2.00  200.00     10.00                  49.00             3.00
75%           45.00      3.00  265.80     15.00                  71.00             4.00
max           60.00      3.00  496.67     15.00                  89.00             5.00


## Section 3 — Univariate Analysis
Understand each variable on its own before looking at relationships.

In [10]:
# Task 3.1 — categorical distributions
print("Top 8 food items ordered:")
print(df_clean['food_item'].value_counts().head(8))

print("\nOrder status breakdown:")
status_pct = (df_clean['order_status'].value_counts(normalize=True) * 100).round(1).astype(str) + '%'
print(status_pct)

Top 8 food items ordered:
food_item
momos       57
Burger      47
Pizza       44
noodles     44
Coke        42
Biryani     41
Sandwich    38
Pasta       37
Name: count, dtype: int64

Order status breakdown:
order_status
Cancelled    41.7%
Completed    38.6%
Returned     19.7%
Name: proportion, dtype: object


In [11]:
# Task 3.2 — numeric distributions
print("Price distribution:")
print(f"  Mean:   ₹{df_clean['price'].mean():.2f}")
print(f"  Median: ₹{df_clean['price'].median():.2f}")
print(f"  Std:    ₹{df_clean['price'].std():.2f}")
print(f"  Skew:   {df_clean['price'].skew():.2f}  → right skewed (a few high-value orders pulling the mean up)")

print("\nDelivery time distribution:")
print(f"  Mean:   {df_clean['delivery_time_minutes'].mean():.1f} mins")
print(f"  Median: {df_clean['delivery_time_minutes'].median():.1f} mins")
print(f"  Std:    {df_clean['delivery_time_minutes'].std():.1f} mins")

print("\nCustomer age distribution:")
df_clean['age_group'] = pd.cut(
    df_clean['customer_age'],
    bins=[0, 25, 35, 45, 100],
    labels=['<25', '25-35', '35-45', '45+']
)
print(df_clean['age_group'].value_counts().sort_index())

Price distribution:
  Mean:   ₹250.73
  Median: ₹200.00
  Std:    ₹93.48
  Skew:   1.28  → right skewed (a few high-value orders pulling the mean up)

Delivery time distribution:
  Mean:   51.5 mins
  Median: 49.0 mins
  Std:    22.1 mins

Customer age distribution:
age_group
<25      112
25-35     22
35-45     84
45+       28
Name: count, dtype: int64


## Section 4 — Outlier Detection
Using the IQR method to identify and handle outliers.

In [12]:
# Task 4.1 — IQR outlier detection on price
Q1  = df_clean['price'].quantile(0.25)
Q3  = df_clean['price'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

outliers = df_clean[
    (df_clean['price'] < lower_fence) | (df_clean['price'] > upper_fence)
]

print("=== IQR Outlier Analysis — Price ===")
print(f"Q1:    ₹{Q1:.2f}")
print(f"Q3:    ₹{Q3:.2f}")
print(f"IQR:   ₹{IQR:.2f}")
print(f"Lower fence: ₹{lower_fence:.2f}")
print(f"Upper fence: ₹{upper_fence:.2f}")
print(f"Outliers found: {len(outliers)} rows ({len(outliers)/df_clean['price'].notna().sum()*100:.1f}% of non-null prices)")
print("\nOutlier orders:")
print(outliers[['order_id', 'food_item', 'price']].head().to_string())

=== IQR Outlier Analysis — Price ===
Q1:    ₹200.00
Q3:    ₹265.80
IQR:   ₹65.80
Lower fence: ₹101.30
Upper fence: ₹364.49
Outliers found: 26 rows (15.3% of non-null prices)

Outlier orders:
    order_id      food_item   price
25     12195          Coke  496.67
47     14591         Pizza  462.12
54     15738        Burger  432.60
79     11022        Burger  432.60
87     15540         momos  432.60


In [13]:
# Task 4.2 — IQR on delivery time
dt = df_clean['delivery_time_minutes'].dropna()
Q1_dt, Q3_dt = dt.quantile(0.25), dt.quantile(0.75)
IQR_dt = Q3_dt - Q1_dt

outliers_dt = df_clean[
    df_clean['delivery_time_minutes'].notna() &
    ((df_clean['delivery_time_minutes'] < Q1_dt - 1.5*IQR_dt) |
     (df_clean['delivery_time_minutes'] > Q3_dt + 1.5*IQR_dt))
]
print("=== IQR Outlier Analysis — Delivery Time ===")
print(f"Lower fence: {Q1_dt - 1.5*IQR_dt:.2f} mins")
print(f"Upper fence:  {Q3_dt + 1.5*IQR_dt:.2f} mins")
print(f"Outliers: {len(outliers_dt)}  — delivery times are within expected range after cleaning")

=== IQR Outlier Analysis — Delivery Time ===
Lower fence: -12.75 mins
Upper fence:  108.75 mins
Outliers: 0  — delivery times are within expected range after cleaning


## Section 5 — Bivariate Analysis
Explore relationships between variables to find patterns.

In [14]:
# Task 5.1 — delivery time by city
city_delivery = df_clean.groupby('customer_city')['delivery_time_minutes'].mean().round(1).sort_values(ascending=False)
print("Average delivery time by city:")
print(city_delivery)
print(f"\nInsight: Hyderabad has the slowest delivery ({city_delivery.iloc[0]} mins avg), Pune the fastest ({city_delivery.iloc[-1]} mins avg)")

Average delivery time by city:
customer_city
Hyderabad    59.6
Mumbai       56.9
Delhi        54.2
Nagpur       48.2
Bangalore    42.6
Pune         38.0
Name: delivery_time_minutes, dtype: float64

Insight: Hyderabad has the slowest delivery (59.6 mins avg), Pune the fastest (38 mins avg)


In [15]:
# Task 5.2 — rating by order status
rating_by_status = df_clean.groupby('order_status')['delivery_rating'].mean().round(2)
print("Average rating by order status:")
print(rating_by_status)
print("\nInteresting: Cancelled orders have HIGHER ratings than completed ones.")
print("This suggests customers rate the experience before outcome, or rating is for the delivery person, not the order.")

Average rating by order status:
order_status
Cancelled    3.47
Completed    2.67
Returned     3.23
Name: delivery_rating, dtype: float64

Interesting: Cancelled orders have HIGHER ratings than completed ones.
This suggests customers rate the experience before outcome, or rating is for the delivery person, not the order.


In [16]:
# Task 5.3 — delivery time by payment method and rating by age group
print("Average delivery time by payment method:")
print(df_clean.groupby('payment_method')['delivery_time_minutes'].mean().round(1).sort_values())

print("\nAverage rating by age group:")
print(df_clean.groupby('age_group', observed=False)['delivery_rating'].mean().round(2))

Average delivery time by payment method:
payment_method
Wallet    46.3
Cash      50.5
Upi       50.5
Card      56.9
Name: delivery_time_minutes, dtype: float64

Average rating by age group:
age_group
<25      3.05
25-35    2.94
35-45    2.93
45+      3.42
Name: delivery_rating, dtype: float64


## Section 6 — Correlation Analysis

In [17]:
# Task 6.1 — correlation matrix
num_cols = ['customer_age', 'quantity', 'price', 'discount', 'delivery_rating', 'delivery_time_minutes']
corr = df_clean[num_cols].corr().round(3)

print("Correlation matrix:")
print(corr.to_string())
print("\nKey observations:")
print("- No strong correlations exist between any pair of variables (all < 0.15)")
print("- Price and delivery_rating have a slight positive correlation (0.109): higher priced orders get slightly better ratings")
print("- Price and delivery_time have a slight negative correlation (-0.093): pricier orders delivered marginally faster")

Correlation matrix:
                       customer_age  quantity  price  discount  delivery_rating  delivery_time_minutes
customer_age                  1.000    -0.068  0.061    -0.034            0.004                  0.016
quantity                     -0.068     1.000  0.061    -0.030            0.033                  0.091
price                         0.061     0.061  1.000     0.083            0.109                 -0.093
discount                     -0.034    -0.030  0.083     1.000            0.048                 -0.047
delivery_rating               0.004     0.033  0.109     0.048            1.000                 -0.030
delivery_time_minutes         0.016     0.091 -0.093    -0.047           -0.030                  1.000

Key observations:
- No strong correlations exist between any pair of variables (all < 0.15)
- Price and delivery_rating have a slight positive correlation (0.109): higher priced orders get slightly better ratings
- Price and delivery_time have a slight negat

## Section 7 — Feature Engineering

In [18]:
# Task 7.1 — create new features
df_clean['revenue']         = df_clean['price'] * df_clean['quantity']
df_clean['discounted_price'] = df_clean['price'] * (1 - df_clean['discount'] / 100)
df_clean['price_tier']      = pd.cut(
    df_clean['price'],
    bins=[0, 200, 300, 10000],
    labels=['Budget', 'Mid', 'Premium']
)

print("New features created: revenue, discounted_price, price_tier")
print("\nRevenue by city:")
print(df_clean.groupby('customer_city')['revenue'].sum().round(0).sort_values(ascending=False))

print("\nPrice tier distribution:")
print(df_clean['price_tier'].value_counts().sort_index())

New features created: revenue, discounted_price, price_tier

Revenue by city:
customer_city
Pune         11984.0
Mumbai       10996.0
Bangalore     9533.0
Delhi         9305.0
Hyderabad     7796.0
Nagpur        6747.0
Name: revenue, dtype: float64

Price tier distribution:
price_tier
Budget      95
Mid         37
Premium     38
Name: count, dtype: int64


In [19]:
# Task 7.2 — compare high vs low value orders
rev_75 = df_clean['revenue'].quantile(0.75)
rev_25 = df_clean['revenue'].quantile(0.25)

high_val = df_clean[df_clean['revenue'] >= rev_75]
low_val  = df_clean[df_clean['revenue'] <= rev_25]

print(f"High value orders (top 25% revenue):")
print(f"  Count: {len(high_val)}")
print(f"  Avg delivery rating: {high_val['delivery_rating'].mean():.2f}")
print(f"  Avg delivery time:   {high_val['delivery_time_minutes'].mean():.1f} mins")

print(f"\nLow value orders (bottom 25% revenue):")
print(f"  Count: {len(low_val)}")
print(f"  Avg delivery rating: {low_val['delivery_rating'].mean():.2f}")
print(f"  Avg delivery time:   {low_val['delivery_time_minutes'].mean():.1f} mins")

print("\nInsight: High value orders get faster delivery AND slightly better ratings.")

High value orders (top 25% revenue):
  Count: 42
  Avg delivery rating: 2.90
  Avg delivery time:   50.2 mins

Low value orders (bottom 25% revenue):
  Count: 42
  Avg delivery rating: 2.38
  Avg delivery time:   58.2 mins

Insight: High value orders get faster delivery AND slightly better ratings.


## Section 8 — Final Business Insights

After a full EDA pass on this food delivery dataset, here are the key findings:

- **Data quality is poor** — 51% of price values were missing, 98 delivery times were negative (-5), ratings went up to 6.0 (invalid scale), and categorical columns had 3-4 different spellings of the same value (e.g. 'upi', 'UPI', 'Upi'). Real-world data always needs this kind of cleaning before any analysis.

- **41.7% of orders were cancelled** — that's a serious problem. Less than 40% of orders actually completed successfully, which would be a major red flag for operations.

- **Hyderabad has the slowest deliveries (59.6 mins avg)** vs Pune at 38 mins. Could indicate operational inefficiencies or longer distances in Hyderabad.

- **Wallet payments correlate with faster delivery (46.3 mins)** vs Card payments (56.9 mins). Could be coincidence but worth investigating — perhaps card payments have more verification delays.

- **No strong correlations between variables** — price, age, quantity, discount, and rating are essentially independent of each other in this dataset. This suggests customer behaviour is not easily predictable from these features alone.

- **Pune generates the most revenue (₹11,984)** despite not having the most orders — suggesting higher average order values in that city.